# CitationRepairEngine F1-F8 production run

This notebook runs the real `production_launcher.launch_full` path. It is not a
demo and it does not substitute synthetic tests for a corpus run.

What it gives you:

- exact, pinned F1-F8 production execution;
- every citing XML and every successfully retrieved cited full text cached in Drive;
- live heartbeats and immediate finding previews while the run is active;
- a reviewer report with paper links, claims, exact cited-paper evidence spans,
  model decisions, verifier decisions, provenance, and the complete raw record;
- publication-oriented measurement tables for reachability, frequency, holds,
  input shape, timing, tokens, estimated cost, caching, and useful comparisons;
- immutable raw engine artifacts kept separate from the human-facing review report.

## Before running

1. In Colab, add secrets named `ANTHROPIC_API_KEY` and `NCBI_API_KEY`.
2. Put four complete F7 authority snapshot JSON files plus their manifest in
   `MyDrive/CitationRepairEngine/f7_authorities/`. Do not use tiny or
   paper-specific snapshots; unresolved authority coverage suppresses genuine F7s.
3. In Cell 2 choose a corpus source. `pmcid_file` is recommended for a paper:
   one PMCID per line in `MyDrive/CitationRepairEngine/pmcids.txt`.
4. Run top to bottom. The paid cell remains locked until you deliberately set
   `ENABLE_PAID_RUN = True`.

The production launcher requires a fresh output directory and deliberately has no
resume mode. Failed attempts remain in Drive; choose a new `RUN_NAME` to retry.

## 1. Install, mount Drive, and load the exact production engine

In [ ]:
# Estimated runtime: 1-3 minutes (package install, clone, and import).
import os
import sys
import re
import json
import time
import html
import math
import hashlib
import shutil
import threading
import subprocess
import statistics
import collections
import xml.etree.ElementTree as ET
from pathlib import Path
from datetime import datetime, timezone
from urllib.parse import quote, urlparse

EXPECTED_COMMIT = "f0f835cb7a897d3e809244450dbbd7201ae53af9"
BRANCH = "merge/f2-into-f3f7"
REPO_URL = "https://github.com/astonliu/citation-repair-engine.git"
REPO = Path("/content/citation-repair-engine")
PKG_ROOT = REPO / "citation_repair_F1_handoff"

subprocess.run([
    sys.executable, "-m", "pip", "-q", "install",
    "rapidfuzz==3.14.5", "requests==2.32.5", "lxml==6.0.2",
    "anthropic==0.122.0", "jsonschema==4.26.0", "pandas==2.2.2",
], check=True)

from google.colab import drive, userdata
drive.mount("/content/drive", force_remount=False)

if any(name == "cre" or name.startswith("cre.f1") for name in sys.modules):
    raise RuntimeError("CRE is already imported. Restart the Colab runtime and rerun.")

if REPO.exists():
    if not (REPO / ".git").exists():
        raise RuntimeError(f"{REPO} exists but is not the expected git checkout")
    dirty = subprocess.check_output(
        ["git", "-C", str(REPO), "status", "--porcelain"], text=True).strip()
    if dirty:
        raise RuntimeError(f"Refusing a dirty Colab checkout:\n{dirty}")
else:
    subprocess.run(["git", "clone", REPO_URL, str(REPO)], check=True)

subprocess.run(["git", "-C", str(REPO), "fetch", "origin", BRANCH], check=True)
known = subprocess.run([
    "git", "-C", str(REPO), "cat-file", "-e", f"{EXPECTED_COMMIT}^{{commit}}"
]).returncode == 0
if not known:
    raise RuntimeError(
        f"The remote branch does not contain required commit {EXPECTED_COMMIT}. "
        "Push merge/f2-into-f3f7, restart the runtime, and rerun this cell.")
subprocess.run([
    "git", "-C", str(REPO), "checkout", "--detach", EXPECTED_COMMIT
], check=True)

HEAD = subprocess.check_output(
    ["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
tracked_dirty = subprocess.check_output([
    "git", "-C", str(REPO), "status", "--porcelain", "--untracked-files=no"
], text=True).strip()
assert HEAD == EXPECTED_COMMIT and not tracked_dirty

sys.path.insert(0, str(PKG_ROOT))
from cre.f1 import (
    band_prompts, coverage_aggregate, coverage_prompts_v3, evidence_reader,
    fulltext_reader, ncbi_meta, parser, preband_contract,
    production_launcher, ratelimit,
)
from cre.f1.recording_adapter import AdapterReceipt, wrap_run_seams

assert callable(production_launcher.launch_full)
assert "f7_evidence_builder.py" in production_launcher.GOVERNING_MODULES
assert "f8_retraction.py" in production_launcher.GOVERNING_MODULES
tree_receipt = production_launcher.verify_tree(str(REPO), str(PKG_ROOT / "cre/f1"))

DRIVE_ROOT = Path("/content/drive/MyDrive/CitationRepairEngine")
CACHE_ROOT = DRIVE_ROOT / "cache"
RUNS_ROOT = DRIVE_ROOT / "runs"
AUTHORITY_ROOT = DRIVE_ROOT / "f7_authorities"
for directory in (CACHE_ROOT, RUNS_ROOT, AUTHORITY_ROOT):
    directory.mkdir(parents=True, exist_ok=True)

print("ENGINE COMMIT:", HEAD)
print("GOVERNED MODULES:", len(production_launcher.GOVERNING_MODULES))
print("DRIVE ROOT:", DRIVE_ROOT)
print("PRODUCTION ENGINE: READY")

## 2. Run configuration and credentials

`pmcid_file` is the cleanest corpus definition for a paper. `query_sample` is
available for a deterministic exploratory corpus; report its query, pool size,
seed, and date, and do not call that a population prevalence sample.

In [ ]:
# Estimated runtime: under 5 seconds.
MODEL = "claude-opus-5"
EMAIL = "aston.hliu@gmail.com"

CORPUS_MODE = "pmcid_file"          # "pmcid_file" or "query_sample"
PMCID_FILE = DRIVE_ROOT / "pmcids.txt"

# Used only when CORPUS_MODE == "query_sample".
PMC_QUERY = '"open access"[filter] AND 2024:2025[pdat]'
QUERY_POOL_SIZE = 2000
PAPERS_PER_RUN = 3                 # small first batch; increase after observed cost
BATCH_INDEX = 0                    # 0, 1, 2... selects disjoint corpus slices
CORPUS_SEED = 20260820

MAX_WORKERS = 4
MODEL_MAX_TOKENS = 2048
ANTHROPIC_MAX_RETRIES = 3
ENABLE_PROMPT_CACHE = True
ENABLE_PAID_RUN = False             # read Cell 7, then change this to True

# Keep RUN_NAME stable while one attempt is running. Use a new name after failure.
RUN_NAME = globals().get("RUN_NAME") or (
    f"f1_f8_batch{BATCH_INDEX:03d}_" +
    datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ"))
SNAPSHOT_DATE = datetime.now(timezone.utc).date().isoformat()

def secret(name, required=True):
    try:
        value = userdata.get(name) or ""
    except Exception:
        value = ""
    if required and not value:
        raise RuntimeError(f"Add {name} under Colab > Secrets, then rerun this cell")
    return value

ANTHROPIC_API_KEY = secret("ANTHROPIC_API_KEY")
NCBI_API_KEY = secret("NCBI_API_KEY")
ratelimit.configure_ncbi(True)

# Claude Opus 5 public API prices, USD per million tokens, checked 2026-08-20.
# Edit these if the linked provider page changes before the run.
PRICE_USD_PER_MTOK = {
    "input_tokens": 5.00,
    "cache_creation_input_tokens": 6.25,
    "cache_read_input_tokens": 0.50,
    "output_tokens": 25.00,
}
PRICE_SOURCE = "https://platform.claude.com/docs/en/about-claude/pricing"
PRICE_CHECKED_DATE = "2026-08-20"

RUN_ROOT = RUNS_ROOT / RUN_NAME
OUTPUT_ROOT = RUN_ROOT / "production_output"
EVENT_ROOT = RUN_ROOT / "events"
MEASURE_ROOT = RUN_ROOT / "measurements"
REVIEW_ROOT = RUN_ROOT / "review"
for directory in (EVENT_ROOT, MEASURE_ROOT, REVIEW_ROOT):
    directory.mkdir(parents=True, exist_ok=True)

print("RUN:", RUN_NAME)
print("CORPUS MODE:", CORPUS_MODE)
print("WORKERS:", MAX_WORKERS)
print("OUTPUT:", OUTPUT_ROOT)
print("Secrets found: ANTHROPIC_API_KEY, NCBI_API_KEY (values hidden)")

## 3. Durable telemetry, HTTP logging, and model-call logging

Telemetry contains hashes, timings, token counts, status codes, and stage names.
It never stores API keys or full prompts/model responses.

In [ ]:
# Estimated runtime: under 5 seconds.
import requests
from anthropic import Anthropic

_FILE_LOCKS_GUARD = threading.Lock()
_FILE_LOCKS = {}

def utc_now():
    return datetime.now(timezone.utc).isoformat()

def sha256_bytes(data):
    return hashlib.sha256(data).hexdigest()

def sha256_file(path):
    return sha256_bytes(Path(path).read_bytes())

def file_lock(path):
    key = str(Path(path))
    with _FILE_LOCKS_GUARD:
        return _FILE_LOCKS.setdefault(key, threading.Lock())

def append_jsonl(path, record):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with file_lock(path), path.open("a", encoding="utf-8") as fh:
        fh.write(json.dumps(record, sort_keys=True, ensure_ascii=False) + "\n")
        fh.flush()
        os.fsync(fh.fileno())

def read_jsonl(path):
    path = Path(path)
    if not path.exists():
        return []
    return [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines()
            if line.strip()]

def atomic_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_suffix(path.suffix + ".tmp")
    temp.write_text(json.dumps(payload, indent=2, sort_keys=True,
                               ensure_ascii=False) + "\n", encoding="utf-8")
    os.replace(temp, path)

HTTP_EVENTS = EVENT_ROOT / "http_events.jsonl"
MODEL_EVENTS = EVENT_ROOT / "model_events.jsonl"
CACHE_EVENTS = EVENT_ROOT / "cache_events.jsonl"

class LoggedThreadLocalSession:
    """A separate requests.Session per worker, with redacted durable telemetry."""
    def __init__(self, stage):
        self.stage = stage
        self.local = threading.local()

    def _client(self):
        if not hasattr(self.local, "client"):
            self.local.client = requests.Session()
        return self.local.client

    def request(self, method, url, **kwargs):
        started = time.perf_counter()
        safe_params = {str(k): str(v)[:300] for k, v in
                       dict(kwargs.get("params") or {}).items()
                       if str(k).lower() not in {"api_key", "key", "token"}}
        event = {"ts": utc_now(), "stage": self.stage, "method": method,
                 "host": urlparse(url).netloc, "path": urlparse(url).path,
                 "params": safe_params}
        try:
            response = self._client().request(method, url, **kwargs)
            event.update(result="response", status_code=response.status_code,
                         elapsed_s=round(time.perf_counter() - started, 4),
                         response_bytes=len(response.content or b""))
            return response
        except Exception as exc:
            event.update(result="exception", exception_type=type(exc).__name__,
                         message=str(exc)[:500],
                         elapsed_s=round(time.perf_counter() - started, 4))
            raise
        finally:
            append_jsonl(HTTP_EVENTS, event)

    def get(self, url, **kwargs):
        return self.request("GET", url, **kwargs)

    def post(self, url, **kwargs):
        return self.request("POST", url, **kwargs)

SESSION = LoggedThreadLocalSession("production_retrieval")

_anthropic_local = threading.local()
def anthropic_client():
    if not hasattr(_anthropic_local, "client"):
        _anthropic_local.client = Anthropic(
            api_key=ANTHROPIC_API_KEY, max_retries=ANTHROPIC_MAX_RETRIES,
            timeout=180.0)
    return _anthropic_local.client

def usage_dict(response):
    usage = getattr(response, "usage", None)
    return {name: int(getattr(usage, name, 0) or 0) for name in
            PRICE_USD_PER_MTOK}

def make_model_transport(stage, *, max_tokens=MODEL_MAX_TOKENS,
                         cache_prefix=None):
    """Distinct, thread-safe callable. Prompt caching changes only transport shape."""
    warm_lock = threading.Lock()
    warmed = threading.Event()

    def transport(prompt):
        started = time.perf_counter()
        prompt_hash = sha256_bytes(prompt.encode("utf-8"))
        use_cache = bool(ENABLE_PROMPT_CACHE and cache_prefix and
                         prompt.startswith(cache_prefix))
        content = prompt
        if use_cache:
            content = [
                {"type": "text", "text": cache_prefix,
                 "cache_control": {"type": "ephemeral", "ttl": "5m"}},
                {"type": "text", "text": prompt[len(cache_prefix):]},
            ]
            assert "".join(block["text"] for block in content) == prompt

        event = {"ts": utc_now(), "stage": stage, "model": MODEL,
                 "prompt_sha256": prompt_hash, "prompt_chars": len(prompt),
                 "max_tokens": max_tokens, "cache_requested": use_cache}

        def send():
            return anthropic_client().messages.create(
                model=MODEL, max_tokens=max_tokens,
                messages=[{"role": "user", "content": content}])

        try:
            if use_cache and not warmed.is_set():
                with warm_lock:
                    if not warmed.is_set():
                        event["cache_warm_request"] = True
                        response = send()
                        warmed.set()
                    else:
                        response = send()
            else:
                response = send()
            text = "".join(block.text for block in response.content
                           if getattr(block, "type", None) == "text")
            event.update(result="success", output_chars=len(text),
                         output_sha256=sha256_bytes(text.encode("utf-8")),
                         elapsed_s=round(time.perf_counter() - started, 4),
                         **usage_dict(response))
            return text
        except Exception as exc:
            event.update(result="exception", exception_type=type(exc).__name__,
                         status_code=getattr(exc, "status_code", None),
                         message=str(exc)[:500],
                         elapsed_s=round(time.perf_counter() - started, 4))
            raise
        finally:
            append_jsonl(MODEL_EVENTS, event)

    transport.model_id = MODEL
    transport.model_settings = {"max_tokens": max_tokens,
                                "prompt_cache": bool(cache_prefix)}
    transport.thread_safe = True
    return transport

print("Telemetry root:", EVENT_ROOT)
print("Prompt cache enabled:", ENABLE_PROMPT_CACHE)

## 4. Define, retrieve, validate, and cache the citing-paper corpus

Every validated JATS XML is written to Drive before the production run. A source
manifest binds filenames to SHA-256 hashes. Re-running this cell uses cached bytes.

In [ ]:
# Estimated runtime: roughly 1-4 seconds per uncached paper; cached papers are local.
CORPUS_CACHE = CACHE_ROOT / "citing_xml"
CORPUS_CACHE.mkdir(parents=True, exist_ok=True)

def normalize_pmcid(value):
    value = str(value).strip().upper()
    if value.isdigit():
        value = "PMC" + value
    if not re.fullmatch(r"PMC\d+", value):
        raise ValueError(f"Invalid PMCID: {value!r}")
    return value

def esearch_pmc(term, retmax):
    params = {"db": "pmc", "term": term, "retmax": retmax, "retmode": "json",
              "tool": "CitationRepairEngine", "email": EMAIL,
              "api_key": NCBI_API_KEY}
    response = SESSION.get(
        "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi",
        params=params, timeout=90)
    response.raise_for_status()
    return [normalize_pmcid(x) for x in
            response.json()["esearchresult"].get("idlist", [])]

if CORPUS_MODE == "pmcid_file":
    if not PMCID_FILE.exists():
        raise RuntimeError(
            f"Create {PMCID_FILE} with one PMCID per line, then rerun this cell")
    pool = list(dict.fromkeys(normalize_pmcid(line) for line in
                PMCID_FILE.read_text(encoding="utf-8").splitlines()
                if line.strip() and not line.lstrip().startswith("#")))
    start = BATCH_INDEX * PAPERS_PER_RUN
    PMCIDS = pool[start:start + PAPERS_PER_RUN]
    selection = {"mode": CORPUS_MODE, "source_file": str(PMCID_FILE),
                 "pool_size": len(pool), "batch_index": BATCH_INDEX,
                 "papers_per_run": PAPERS_PER_RUN}
elif CORPUS_MODE == "query_sample":
    pool = sorted(set(esearch_pmc(PMC_QUERY, QUERY_POOL_SIZE)))
    if len(pool) < PAPERS_PER_RUN:
        raise RuntimeError(f"Query returned only {len(pool)} unique PMCIDs")
    ordered = sorted(pool, key=lambda p: hashlib.sha256(
        f"{CORPUS_SEED}:{p}".encode()).hexdigest())
    start = BATCH_INDEX * PAPERS_PER_RUN
    PMCIDS = ordered[start:start + PAPERS_PER_RUN]
    selection = {"mode": CORPUS_MODE, "query": PMC_QUERY,
                 "query_pool_requested": QUERY_POOL_SIZE,
                 "query_pool_observed": len(pool),
                 "papers_per_run": PAPERS_PER_RUN,
                 "batch_index": BATCH_INDEX,
                 "seed": CORPUS_SEED, "selected_at": utc_now()}
else:
    raise ValueError("CORPUS_MODE must be 'pmcid_file' or 'query_sample'")

PMCIDS = list(dict.fromkeys(PMCIDS))
if not PMCIDS:
    raise RuntimeError(
        "This batch is empty. Add more PMCIDs, enlarge the query pool, or stop.")

def validate_jats(payload):
    if len(payload) < 500:
        raise ValueError(f"implausibly small XML ({len(payload)} bytes)")
    root = ET.fromstring(payload)
    tags = [node.tag.rsplit("}", 1)[-1] for node in root.iter()
            if isinstance(node.tag, str)]
    if "article" not in tags:
        raise ValueError("no JATS article element")

def fetch_citing_xml(pmcid):
    path = CORPUS_CACHE / f"{pmcid}.xml"
    if path.exists():
        payload = path.read_bytes()
        validate_jats(payload)
        append_jsonl(CACHE_EVENTS, {"ts": utc_now(), "kind": "citing_xml",
                                   "key": pmcid, "result": "hit"})
        return path, "cache"
    response = SESSION.get(
        "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi",
        params={"db": "pmc", "id": pmcid[3:], "retmode": "xml",
                "tool": "CitationRepairEngine", "email": EMAIL,
                "api_key": NCBI_API_KEY}, timeout=120)
    response.raise_for_status()
    validate_jats(response.content)
    path.write_bytes(response.content)
    append_jsonl(CACHE_EVENTS, {"ts": utc_now(), "kind": "citing_xml",
                               "key": pmcid, "result": "write",
                               "bytes": len(response.content)})
    return path, "live"

corpus_id = sha256_bytes("\n".join(PMCIDS).encode())[:16]
CORPUS_ROOT = DRIVE_ROOT / "corpora" / f"corpus_{corpus_id}"
XML_DIR = CORPUS_ROOT / "xml"
XML_DIR.mkdir(parents=True, exist_ok=True)

retrieval_sources = collections.Counter()
for index, pmcid in enumerate(PMCIDS, 1):
    cached, source = fetch_citing_xml(pmcid)
    retrieval_sources[source] += 1
    active = XML_DIR / cached.name
    if active.exists():
        if active.read_bytes() != cached.read_bytes():
            raise RuntimeError(f"Corpus bytes changed for {pmcid}")
    else:
        shutil.copy2(cached, active)
    if index == 1 or index % 10 == 0 or index == len(PMCIDS):
        print(f"[corpus] {index}/{len(PMCIDS)} cached={retrieval_sources['cache']} "
              f"downloaded={retrieval_sources['live']}", flush=True)

CORPUS_MANIFEST = CORPUS_ROOT / "corpus_manifest.json"
if CORPUS_MANIFEST.exists():
    manifest_payload = json.loads(CORPUS_MANIFEST.read_text(encoding="utf-8"))
    preband_contract.verify_corpus_contents(
        str(XML_DIR), preband_contract.corpus_inventory(manifest_payload))
else:
    manifest_payload = preband_contract.build_corpus_manifest(
        str(XML_DIR), str(CORPUS_MANIFEST))

selection.update({"pmcids": PMCIDS, "pmcid_set_sha256":
                  sha256_bytes("\n".join(sorted(PMCIDS)).encode()),
                  "corpus_manifest": str(CORPUS_MANIFEST),
                  "corpus_manifest_sha256": sha256_file(CORPUS_MANIFEST)})
atomic_json(CORPUS_ROOT / "selection.json", selection)

profiles = []
all_refs = []
for pmcid in PMCIDS:
    refs = parser.parse_pmc_xml(str(XML_DIR / f"{pmcid}.xml"),
                                source_pmcid=pmcid)
    all_refs.extend(refs)
    profiles.append({
        "pmcid": pmcid, "references": len(refs),
        "references_with_pmid": sum(bool(r.claimed.claimed_pmid) for r in refs),
        "references_with_doi": sum(bool(r.claimed.claimed_doi) for r in refs),
        "references_without_citance": sum(not bool(r.citance) for r in refs),
        "co_cited_references": sum(len(r.citance_group_members or []) > 1 for r in refs),
        "xml_bytes": (XML_DIR / f"{pmcid}.xml").stat().st_size,
    })
atomic_json(MEASURE_ROOT / "input_profile.json", {
    "selection": selection, "papers": profiles,
    "papers_count": len(PMCIDS), "references_count": len(all_refs)})

print("CORPUS HASH:", corpus_id)
print("PAPERS:", len(PMCIDS))
print("REFERENCES:", len(all_refs))
print("AVERAGE REFERENCES/PAPER:", round(len(all_refs) / len(PMCIDS), 2))
print("CITING XML CACHE:", dict(retrieval_sources))

## 5. Load the four complete, frozen F7 authority snapshots

The file `f7_authorities/manifest.json` must have a `sources` array with one
entry each for `gene`, `variant`, `drug`, and `disease`. Each entry contains:
`entity_type`, `authority`, `version`, `lookup_date`, `path`, `sha256`, and
`accept_synonym_as_equivalent`. Paths may be relative to the manifest.

Required authorities: HGNC, ClinVar, RxNorm, and MONDO. These are data files,
not API keys. The notebook refuses missing, partial, stale, or hash-mismatched
snapshots because a small convenience snapshot would silently destroy F7 recall.

In [ ]:
# Estimated runtime: seconds to a few minutes, depending on snapshot size and Drive speed.
from cre.f1.f7_seams import (
    AuthoritySnapshotSource, FrozenAuthorityNormalizer,
    make_production_f7_policy, make_production_f7_seams,
    validate_production_f7_configuration,
)
from cre.f1.f7_evidence_builder import make_production_f7_evidence_builder

AUTHORITY_MANIFEST = AUTHORITY_ROOT / "manifest.json"
if not AUTHORITY_MANIFEST.exists():
    raise RuntimeError(
        f"Missing {AUTHORITY_MANIFEST}. Supply complete HGNC, ClinVar, RxNorm, "
        "and MONDO snapshots plus the manifest described above. Do not use demo data.")
authority_manifest = json.loads(AUTHORITY_MANIFEST.read_text(encoding="utf-8"))
source_rows = authority_manifest.get("sources")
if not isinstance(source_rows, list) or len(source_rows) != 4:
    raise RuntimeError("F7 authority manifest must contain exactly four sources")

authority_sources = []
for row in source_rows:
    path = Path(row["path"])
    if not path.is_absolute():
        path = AUTHORITY_MANIFEST.parent / path
    if not path.exists():
        raise RuntimeError(f"Missing authority snapshot: {path}")
    observed_hash = sha256_file(path)
    if observed_hash != row["sha256"]:
        raise RuntimeError(f"Authority SHA-256 mismatch: {path.name}")
    authority_sources.append(AuthoritySnapshotSource(
        entity_type=row["entity_type"], authority=row["authority"],
        version=row["version"], lookup_date=row["lookup_date"],
        path=str(path), sha256=row["sha256"],
        accept_synonym_as_equivalent=row["accept_synonym_as_equivalent"]))

F7_NORMALIZER = FrozenAuthorityNormalizer(authority_sources)
print(json.dumps(F7_NORMALIZER.source_manifest(), indent=2, sort_keys=True))
print("F7 AUTHORITIES: VALID")

## 6. Wire the real production seams and validate reachability

Successful cited-paper abstracts, full texts, PubMed metadata, reference lists,
PMCID resolutions, and publication types are persisted under `cache/` in Drive.
Failures are not cached as negatives.

In [ ]:
# Estimated runtime: under 10 seconds; no model call is made.
from cre.f1 import f5_seams as f5s
from cre.f1.f5_supersession import F5Policy

ABSTRACT_CACHE = CACHE_ROOT / "abstracts"
FULLTEXT_CACHE = CACHE_ROOT / "fulltexts"
F5_CACHE = CACHE_ROOT / "f5_pubmed"
LOOKUP_CACHE = CACHE_ROOT / "lookups"
for directory in (ABSTRACT_CACHE, FULLTEXT_CACHE, F5_CACHE, LOOKUP_CACHE):
    directory.mkdir(parents=True, exist_ok=True)

_lookup_locks_guard = threading.Lock()
_lookup_locks = {}
def disk_cached(namespace, key, producer):
    key = str(key or "").strip()
    digest = sha256_bytes(key.encode())
    path = LOOKUP_CACHE / namespace / f"{digest}.json"
    with _lookup_locks_guard:
        lock = _lookup_locks.setdefault((namespace, key), threading.Lock())
    with lock:
        if path.exists():
            append_jsonl(CACHE_EVENTS, {"ts": utc_now(), "kind": namespace,
                                       "key_sha256": digest, "result": "hit"})
            return json.loads(path.read_text(encoding="utf-8"))["value"]
        value = producer()
        if value is not None:
            path.parent.mkdir(parents=True, exist_ok=True)
            atomic_json(path, {"key": key, "value": value, "cached_at": utc_now()})
            append_jsonl(CACHE_EVENTS, {"ts": utc_now(), "kind": namespace,
                                       "key_sha256": digest, "result": "write"})
        else:
            append_jsonl(CACHE_EVENTS, {"ts": utc_now(), "kind": namespace,
                                       "key_sha256": digest, "result": "miss_not_cached"})
        return value

def fetch_abstract_cached(pmid):
    before = evidence_reader._read_cache(str(ABSTRACT_CACHE), str(pmid))
    value = evidence_reader.fetch_abstract(
        str(pmid), api_key=NCBI_API_KEY, email=EMAIL, session=SESSION,
        cache_dir=str(ABSTRACT_CACHE))
    append_jsonl(CACHE_EVENTS, {"ts": utc_now(), "kind": "abstract",
                               "key": str(pmid),
                               "result": "hit" if before is not None else
                               ("write" if value is not None else "miss_not_cached")})
    return value

def fetch_fulltext_cached(pmid):
    cache_path = Path(fulltext_reader._cache_path(str(FULLTEXT_CACHE), str(pmid)))
    existed = cache_path.exists()
    value = fulltext_reader.fetch_fulltext(
        str(pmid), api_key=NCBI_API_KEY, email=EMAIL, session=SESSION,
        cache_dir=str(FULLTEXT_CACHE))
    append_jsonl(CACHE_EVENTS, {"ts": utc_now(), "kind": "fulltext",
                               "key": str(pmid),
                               "result": "hit" if existed else
                               ("write" if value is not None else "miss_not_cached"),
                               "complete": isinstance(value, dict) and
                               value.get("retrieval_complete") is True})
    return value

def fetch_reflist_cached(pmcid):
    return disk_cached("reflist", pmcid, lambda: ncbi_meta.ncbi_pmc_reflist(
        str(pmcid), api_key=NCBI_API_KEY, email=EMAIL, session=SESSION))

def resolve_pmcid_cached(pmid):
    return disk_cached("pmcid", pmid, lambda: fulltext_reader._live_resolve_pmcid(
        str(pmid), NCBI_API_KEY, EMAIL, SESSION))

def pubtypes_cached(pmid):
    return disk_cached("pubtypes", pmid, lambda: ncbi_meta.ncbi_pubtypes(
        str(pmid), NCBI_API_KEY, EMAIL, session=SESSION))

coverage_prefix = coverage_prompts_v3.COVERAGE_PROMPT_V3.partition(
    "<<ATOMIC_CLAIM>>")[0]
transports = {
    "band1": make_model_transport("band1", max_tokens=400),
    "extractor": make_model_transport("claim_extraction"),
    "abstract_coverage": make_model_transport("abstract_coverage"),
    "fulltext_coverage": make_model_transport(
        "fulltext_coverage", cache_prefix=coverage_prefix),
    "discriminator": make_model_transport("f3_f4_discriminator"),
    "f4_verifier": make_model_transport("f4_verifier"),
    "f5_generator": make_model_transport("f5_generator"),
    "f5_verifier": make_model_transport("f5_verifier"),
    "f7_generator": make_model_transport("f7_generator"),
    "f7_verifier": make_model_transport("f7_verifier"),
}

F5_SOURCE_PACKETS = []
F5_THIN_SOURCES = []
F5_SPAN_MISSES = []
F5_PROTOCOLS = []

receipt = AdapterReceipt(model=MODEL, temperature="unsupported",
                         assistant_prefill="unsupported")
run_seams = wrap_run_seams(
    receipt,
    extractor=band_prompts.make_extractor(transports["extractor"]),
    coverage_judge=coverage_aggregate.make_coverage_judge(
        transports["abstract_coverage"]),
    coverage_judge_v3=coverage_prompts_v3.make_coverage_judge_v3(
        transports["fulltext_coverage"]),
    fetch_abstract=fetch_abstract_cached,
    fetch_fulltext=fetch_fulltext_cached,
    fetch_reflist=fetch_reflist_cached,
    discriminator_call_llm=transports["discriminator"],
    f4_verifier_call_llm=transports["f4_verifier"],
    f3_fetch_reflist=fetch_reflist_cached,
    f3_resolve_pmcid=resolve_pmcid_cached,
    pubtypes_lookup=pubtypes_cached,
)

f5_runtime = f5s.build_pubmed_f5_runtime(
    complete=transports["f5_generator"],
    verifier_complete=transports["f5_verifier"],
    as_of_date=SNAPSHOT_DATE, api_key=NCBI_API_KEY, email=EMAIL,
    session=SESSION, cache_dir=str(F5_CACHE),
    fetch_fulltext=fetch_fulltext_cached,
    source_packet_log=F5_SOURCE_PACKETS,
    thin_source_log=F5_THIN_SOURCES,
    span_miss_log=F5_SPAN_MISSES,
    protocol_log=F5_PROTOCOLS,
    judgment_cache={}, judgment_model_id=MODEL, verifier_model_id=MODEL,
    judgment_model_settings={"max_tokens": MODEL_MAX_TOKENS})
F5_POLICY = F5Policy(mode="deployment", deploy_path_a=False,
                     generator_model_id=MODEL, verifier_model_id=MODEL)

F7_EVIDENCE_BUILDER = make_production_f7_evidence_builder()
F7_POLICY = make_production_f7_policy(
    F7_NORMALIZER, generator_model_id=MODEL, verifier_model_id=MODEL)
F7_SEAMS = make_production_f7_seams(
    generator_transport=transports["f7_generator"],
    verifier_transport=transports["f7_verifier"],
    normalizer=F7_NORMALIZER, adapter_receipt=receipt,
    max_parallel=MAX_WORKERS)

f5s.validate_production_f5_configuration(
    seams=f5_runtime["f5_seams"],
    evidence_builder=f5_runtime["f5_evidence_builder"],
    policy=F5_POLICY, run_model=MODEL)
validate_production_f7_configuration(
    seams=F7_SEAMS, evidence_builder=F7_EVIDENCE_BUILDER,
    policy=F7_POLICY, adapter_receipt=receipt)

production_launcher.verify_temperature_governance(model=MODEL, temperature=None)
production_launcher.verify_prefill_governance(model=MODEL, assistant_prefill="")
preband_contract.verify_corpus_contents(
    str(XML_DIR), preband_contract.corpus_inventory(
        json.loads(CORPUS_MANIFEST.read_text(encoding="utf-8"))))

REACHABILITY_PREFLIGHT = {
    "F1": "wired: current Band-1 PMID/DOI existence",
    "F2": "wired: current Band-1 identity mismatch",
    "F3": "wired: provenance discriminator + reference-list/PMCID seams",
    "F4": "wired: strength generator + distinct verifier callable",
    "F5": "wired: PubMed candidate/evidence runtime + distinct verifier callable",
    "F6": "wired: complete-fulltext coverage with exact evidence spans",
    "F7": "wired: production evidence builder + four frozen authorities + verifier",
    "F8": "wired: retraction notice/date timing gate in current Band 1",
}
for label, status in REACHABILITY_PREFLIGHT.items():
    print(label, status)
print("ALL F1-F8 PRODUCTION SEAMS: READY")

## 7. Paid-run gate and live feedback

The exact cost is unknown until the model returns token usage. During the run,
this watcher prints a heartbeat every 30 seconds and immediately prints each
genuine machine finding with links, the citing sentence, claims, evidence spans,
and decision fields. Full reviewer cards are generated after completion.

In [ ]:
# Estimated runtime: continuous while the production cell runs.
from IPython.display import display, HTML, FileLink

def paper_links(record):
    links = []
    citing_pmcid = str(record.get("citing_pmcid") or
                       record.get("source_pmcid") or "")
    cited_pmid = str(record.get("cited_pmid") or "")
    claimed = record.get("cited_claimed") or record.get("claimed") or {}
    doi = str(claimed.get("claimed_doi") or claimed.get("doi") or "")
    if citing_pmcid:
        links.append(("citing full paper",
                      f"https://pmc.ncbi.nlm.nih.gov/articles/{citing_pmcid}/"))
    if cited_pmid:
        links.append(("cited paper", f"https://pubmed.ncbi.nlm.nih.gov/{cited_pmid}/"))
    if doi:
        links.append(("cited DOI", f"https://doi.org/{quote(doi, safe='/:')}"))
    return links

def labels_of(record):
    labels = set()
    label = record.get("label")
    if isinstance(label, str) and re.fullmatch(r"F[1-8]", label):
        labels.add(label)
    for value in record.get("findings") or []:
        if isinstance(value, str) and re.fullmatch(r"F[1-8]", value):
            labels.add(value)
        elif isinstance(value, dict):
            derived = value.get("derived") or value.get("label")
            if isinstance(derived, str) and re.fullmatch(r"F[1-8]", derived):
                labels.add(derived)
    if any((row or {}).get("derived") == "F4"
           for row in record.get("strength_records") or []):
        labels.add("F4")
    if any((row or {}).get("derived") == "F5"
           for row in record.get("f5_records") or []):
        labels.add("F5")
    if any((row or {}).get("derived") == "F7"
           for row in record.get("f7_records") or []):
        labels.add("F7")
    return sorted(labels)

def exact_spans(record):
    spans = []
    for index, verdict in enumerate(record.get("coverage_verdicts") or []):
        for span in verdict.get("evidence_spans") or []:
            if isinstance(span, dict) and span.get("text"):
                spans.append({"claim_index": index, "section": span.get("label"),
                              "text": span.get("text"),
                              "sentence_ids": span.get("sentence_ids")})
    return spans

def related_paper_links(obj):
    """Find source-bound PMID/PMCID/DOI fields, including F3/F5/F7 records."""
    found = set()
    def walk(value, key=""):
        if isinstance(value, dict):
            for child_key, child in value.items():
                walk(child, str(child_key))
        elif isinstance(value, list):
            for child in value:
                walk(child, key)
        elif isinstance(value, str):
            folded = key.casefold()
            if "pmid" in folded and value.isdigit():
                found.add((f"PubMed {value}",
                           f"https://pubmed.ncbi.nlm.nih.gov/{value}/"))
            elif "pmcid" in folded and re.fullmatch(r"PMC\d+", value):
                found.add((f"PMC {value}",
                           f"https://pmc.ncbi.nlm.nih.gov/articles/{value}/"))
            elif "doi" in folded and value.startswith("10."):
                found.add((f"DOI {value}",
                           f"https://doi.org/{quote(value, safe='/:')}"))
    walk(obj)
    return sorted(found)

def compact_finding(record, labels):
    return {
        "labels": labels,
        "citation_id": record.get("citation_id"),
        "links": dict(paper_links(record) + related_paper_links(record)),
        "citing_sentence": record.get("citing_sentence") or record.get("citance"),
        "claimed_reference": record.get("cited_claimed") or record.get("claimed"),
        "retrieved_reference": record.get("retrieved"),
        "atomic_claims": record.get("atomic_claims"),
        "exact_cited_paper_spans": exact_spans(record),
        "coverage_decisions": record.get("coverage_verdicts"),
        "provenance_decision": record.get("provenance"),
        "strength_decisions": record.get("strength_records"),
        "f5_decisions": record.get("f5_records"),
        "f7_decisions": record.get("f7_records"),
        "hold_reasons": record.get("hold_reasons"),
    }

WATCH_STATE = {"rows": 0, "findings": collections.Counter(),
               "started": time.time(), "positions": {}}
WATCH_STOP = threading.Event()

def watch_file(path, kind):
    path = Path(path)
    position = WATCH_STATE["positions"].get(str(path), 0)
    if not path.exists():
        return
    with path.open("rb") as fh:
        fh.seek(position)
        while True:
            raw = fh.readline()
            if not raw or not raw.endswith(b"\n"):
                break
            position = fh.tell()
            try:
                record = json.loads(raw.decode("utf-8"))
            except Exception:
                continue
            WATCH_STATE["rows"] += 1
            labels = labels_of(record)
            if (kind == "band1" and
                    record.get("citation_id") in globals().get(
                        "_band1_live_seen", set())):
                continue
            if labels:
                for label in labels:
                    WATCH_STATE["findings"][label] += 1
                packet = compact_finding(record, labels)
                print("\n" + "=" * 80, flush=True)
                print(f">>> GENUINE TAXONOMY CANDIDATE {'+'.join(labels)} "
                      f"{record.get('citation_id')}", flush=True)
                print(json.dumps(packet, indent=2, ensure_ascii=False)[:12000],
                      flush=True)
                append_jsonl(EVENT_ROOT / "live_findings.jsonl",
                             {"kind": kind, "packet": packet, "raw": record})
    WATCH_STATE["positions"][str(path)] = position

def live_watch():
    last_heartbeat = 0.0
    while not WATCH_STOP.is_set():
        watch_file(OUTPUT_ROOT / "band1" / "band1_lossless_log.jsonl", "band1")
        watch_file(OUTPUT_ROOT / "judgment" / "judgment_predictions.jsonl", "band2")
        now = time.time()
        if now - last_heartbeat >= 30:
            last_heartbeat = now
            model_events = read_jsonl(MODEL_EVENTS)
            successful = sum(row.get("result") == "success" for row in model_events)
            print(f"[live] elapsed={(now-WATCH_STATE['started'])/60:.1f} min | "
                  f"records={WATCH_STATE['rows']} | model_calls={successful} | "
                  f"findings={dict(WATCH_STATE['findings'])}", flush=True)
        WATCH_STOP.wait(1.0)

print("PRICE TABLE:", PRICE_USD_PER_MTOK)
print("PRICE SOURCE:", PRICE_SOURCE, "checked", PRICE_CHECKED_DATE)
print("PAPERS:", len(PMCIDS), "REFERENCES:", len(all_refs))
print("No fixed dollar prediction is shown before observed token usage exists.")
if ENABLE_PAID_RUN is not True:
    raise RuntimeError(
        "PAID GATE CLOSED. Review the corpus size and wiring above, then set "
        "ENABLE_PAID_RUN = True in Section 2, rerun Sections 2 and 7, then run "
        "Section 8. If you changed anything besides the gate, rerun Sections 2-7.")

## 8. Execute the real full production launch

Do not call `run_natural_judgment` directly. The launcher binds current Band 1
to Band 2, enforces reportability, and refuses configuration defects before
output creation. It runs the entire frozen corpus; `max_docs` and resume are
intentionally forbidden.

In [ ]:
# Estimated runtime: corpus dependent. Live output appears every 30 seconds.
assert ENABLE_PAID_RUN is True
if OUTPUT_ROOT.exists():
    raise RuntimeError(
        f"Production output already exists: {OUTPUT_ROOT}. Use a new RUN_NAME; "
        "never overwrite or resume a production attempt.")

DEC_069 = {
    "decision_id": "DEC-069", "date": "2026-08-15",
    "section": "PREREGISTRATION.md §6",
    "ruling": (
        "§6 is titled 'Generation-Mode evaluation' and its different-family-judge "
        "commitment governs the judging of GENERATED candidates. The first paper "
        "has no generation mode (DEC-046A defers repair and generation), so §6 "
        "has nothing to bind and the F3-F7 detection band is outside its scope."),
    "residual_risk": (
        "A reviewer may read §6 as covering any LLM-as-judge step. Inside the band, "
        "claude-opus-5 judges coverage of claims claude-opus-5 itself extracted. "
        "That conflict is handled by human adjudication of the machine-positive sample."),
}

# Read-only Band-1 return observer. It does not replace or wrap any engine
# function and it disables itself the instant Band 1 returns. This makes F1/F2/F8
# visible at the moment process_reference finishes instead of waiting for the
# lossless JSONL batch write at the end of Band 1.
from cre.f1 import run as band1_run
_band1_process_code = band1_run.process_reference.__code__
_band1_run_code = band1_run.run.__code__
_band1_live_seen = set()

def band1_live_profile(frame, event, arg):
    if event != "return":
        return band1_live_profile
    if frame.f_code is _band1_process_code and arg is not None:
        try:
            record = arg.to_log_record()
            labels = labels_of(record)
            if labels and record.get("citation_id") not in _band1_live_seen:
                _band1_live_seen.add(record.get("citation_id"))
                for label in labels:
                    WATCH_STATE["findings"][label] += 1
                packet = compact_finding(record, labels)
                print("\n" + "=" * 80, flush=True)
                print(f">>> LIVE GENUINE TAXONOMY CANDIDATE {'+'.join(labels)} "
                      f"{record.get('citation_id')}", flush=True)
                print(json.dumps(packet, indent=2, ensure_ascii=False)[:12000],
                      flush=True)
                append_jsonl(EVENT_ROOT / "live_findings.jsonl",
                             {"kind": "band1_live", "packet": packet,
                              "raw": record})
        except Exception as exc:
            append_jsonl(EVENT_ROOT / "live_observer_errors.jsonl", {
                "ts": utc_now(), "exception_type": type(exc).__name__,
                "message": str(exc)[:500]})
    elif frame.f_code is _band1_run_code:
        sys.setprofile(None)
        print("[live] Band 1 complete; continuing record-by-record Band 2 watch.",
              flush=True)
    return band1_live_profile

WATCH_STATE["started"] = time.time()
WATCH_STOP.clear()
watcher = threading.Thread(target=live_watch, daemon=True)
watcher.start()
run_started = time.perf_counter()
sys.setprofile(band1_live_profile)
try:
    MANIFEST = production_launcher.launch_full(
        repo_dir=str(REPO), pkg_dir=str(PKG_ROOT / "cre/f1"),
        xml_dir=str(XML_DIR), out_dir=str(OUTPUT_ROOT),
        corpus_manifest_path=str(CORPUS_MANIFEST),
        model=MODEL, authorized_models=[MODEL], adapter_receipt=receipt,
        band1_snapshot_date=SNAPSHOT_DATE,
        preregistration_scope_ruling=DEC_069,
        temperature=None, assistant_prefill="",
        anthropic_key=ANTHROPIC_API_KEY, ncbi_key=NCBI_API_KEY,
        crossref_mailto=EMAIL, openalex_mailto=EMAIL,
        f1_complete=transports["band1"],
        f5_seams=f5_runtime["f5_seams"],
        f5_evidence_builder=f5_runtime["f5_evidence_builder"],
        f5_policy=F5_POLICY,
        f7_seams=F7_SEAMS, f7_evidence_builder=F7_EVIDENCE_BUILDER,
        f7_policy=F7_POLICY,
        f4_verifier_model_id=MODEL,
        email=EMAIL, api_key=NCBI_API_KEY, session=SESSION,
        max_workers=MAX_WORKERS,
        **run_seams,
    )
finally:
    sys.setprofile(None)
    RUN_ELAPSED_S = time.perf_counter() - run_started
    WATCH_STOP.set()
    watcher.join(timeout=5)
    atomic_json(EVENT_ROOT / "run_timing.json", {
        "run_name": RUN_NAME, "elapsed_s": RUN_ELAPSED_S,
        "finished_at": utc_now(), "completed": "MANIFEST" in globals()})

assert MANIFEST.get("status") == "complete"
assert (MANIFEST.get("full_launch") or {}).get("all_taxonomies_wired") is True
atomic_json(EVENT_ROOT / "f5_runtime_observations.json", {
    "source_packets": F5_SOURCE_PACKETS,
    "thin_sources": F5_THIN_SOURCES,
    "span_misses": F5_SPAN_MISSES,
    "protocols": F5_PROTOCOLS,
})
print("PRODUCTION RUN COMPLETE in", round(RUN_ELAPSED_S / 60, 2), "minutes")
print("MANIFEST:", MANIFEST["manifest_path"])

## 9. Integrity, reachability, and taxonomy funnel

Reachability is printed before finding counts. A zero is never treated as proof
that a taxonomy ran.

In [ ]:
# Estimated runtime: under 30 seconds.
MANIFEST_PATH = Path(MANIFEST["manifest_path"])
PREDICTIONS_PATH = Path(MANIFEST["predictions_path"])
BAND1_LOG_PATH = OUTPUT_ROOT / "band1" / "band1_lossless_log.jsonl"

manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
band1_rows = read_jsonl(BAND1_LOG_PATH)
band2_rows = read_jsonl(PREDICTIONS_PATH)

assert manifest.get("status") == "complete"
assert manifest.get("accounting_ok") is True
assert manifest.get("module_sha256_stable") is True
assert (manifest.get("executed_domain") or {}).get("matches_preflight") is True
assert len({row.get("citation_id") for row in band2_rows}) == len(band2_rows)
assert len(band1_rows) == len(band2_rows) == len(all_refs)

finding_counts = collections.Counter()
for row in band1_rows:
    if row.get("label") in {"F1", "F2", "F8"}:
        finding_counts[row["label"]] += 1
for row in band2_rows:
    for label in labels_of(row):
        if label in {"F3", "F4", "F5", "F6", "F7"}:
            finding_counts[label] += 1

attest = ((manifest.get("full_launch") or {}).get("band1_check_attestations") or {})
seam_status = manifest.get("seam_status") or {}
reached = {
    "F1": int((attest.get("F1") or {}).get("answered") or 0),
    "F2": int((attest.get("F2") or {}).get("answered") or 0),
    "F3": sum(row.get("provenance") is not None for row in band2_rows),
    "F4": int((manifest.get("f4") or {}).get("eligible_claims") or 0),
    "F5": len([record for row in band2_rows for record in row.get("f5_records") or []]),
    "F6": sum(len(row.get("coverage_verdicts") or []) for row in band2_rows),
    "F7": len([record for row in band2_rows for record in row.get("f7_records") or []]),
    "F8": int((attest.get("F8") or {}).get("answered") or 0),
}

reachability_rows = []
print("REACHABILITY BEFORE COUNTS")
for label in [f"F{i}" for i in range(1, 9)]:
    configured = True
    if label in seam_status and isinstance(seam_status[label], dict):
        configured = seam_status[label].get("wired", True) is True
    status = "RAN" if reached[label] > 0 else "WIRED BUT NOT REACHED"
    reachability_rows.append({"taxonomy": label, "configured": configured,
                              "reached_units": reached[label],
                              "findings": finding_counts[label], "status": status})
    print(f"{label}: configured={configured} | reached={reached[label]} | "
          f"status={status} | findings={finding_counts[label]}")

atomic_json(MEASURE_ROOT / "reachability.json", reachability_rows)
print("INTEGRITY: PASS")

## 10. Reviewer-ready finding report

This is deliberately separate from the blind annotation queue. It exposes the
machine decision because its purpose is to let you judge whether the machine is
right. Every card also contains the complete raw durable record.

In [ ]:
# Estimated runtime: seconds to a few minutes, depending on finding count.
def collect_candidate_ids(obj):
    found = set()
    def walk(value, key=""):
        if isinstance(value, dict):
            for k, v in value.items():
                walk(v, str(k))
        elif isinstance(value, list):
            for v in value:
                walk(v, key)
        elif isinstance(value, str):
            if "pmid" in key.casefold() and value.isdigit():
                found.add(("PubMed " + value,
                           f"https://pubmed.ncbi.nlm.nih.gov/{value}/"))
            elif "pmcid" in key.casefold() and re.fullmatch(r"PMC\d+", value):
                found.add(("PMC " + value,
                           f"https://pmc.ncbi.nlm.nih.gov/articles/{value}/"))
            elif "doi" in key.casefold() and value.startswith("10."):
                found.add(("DOI " + value, f"https://doi.org/{quote(value, safe='/:')}"))
    walk(obj)
    return sorted(found)

review_records = []
for row in band1_rows:
    labels = labels_of(row)
    if labels:
        review_records.append({"source": "band1", "labels": labels, "record": row})
for row in band2_rows:
    labels = [label for label in labels_of(row) if label in {"F3", "F4", "F5", "F6", "F7"}]
    if labels:
        review_records.append({"source": "band2", "labels": labels, "record": row})

REVIEW_JSONL = REVIEW_ROOT / "machine_findings_full_records.jsonl"
with REVIEW_JSONL.open("w", encoding="utf-8") as fh:
    for item in review_records:
        fh.write(json.dumps(item, sort_keys=True, ensure_ascii=False) + "\n")

def esc(value):
    return html.escape(str(value or ""))

def pretty(value):
    return esc(json.dumps(value, indent=2, ensure_ascii=False, sort_keys=True))

cards = []
for index, item in enumerate(review_records, 1):
    record = item["record"]
    labels = item["labels"]
    links = list(dict(paper_links(record)).items()) + collect_candidate_ids(record)
    link_html = " | ".join(
        f'<a href="{esc(url)}" target="_blank">{esc(name)}</a>'
        for name, url in dict(links).items())
    spans = exact_spans(record)
    claimed = record.get("cited_claimed") or record.get("claimed") or {}
    retrieved = record.get("retrieved") or {}
    cards.append(f"""
    <article class="finding">
      <h2>{index}. {esc('+'.join(labels))} — {esc(record.get('citation_id'))}</h2>
      <p class="links">{link_html}</p>
      <h3>Citing sentence</h3><blockquote>{esc(record.get('citing_sentence'))}</blockquote>
      <h3>Reference identity</h3>
      <div class="cols"><pre>{pretty({'claimed': claimed})}</pre><pre>{pretty({'retrieved': retrieved})}</pre></div>
      <h3>Atomic claims</h3><pre>{pretty(record.get('atomic_claims') or [])}</pre>
      <h3>Exact sentences selected from the cited paper</h3><pre>{pretty(spans)}</pre>
      <h3>AI and verifier decisions</h3><pre>{pretty({
          'coverage': record.get('coverage_verdicts'),
          'provenance_F3': record.get('provenance'),
          'strength_F4': record.get('strength_records'),
          'supersession_F5': record.get('f5_records'),
          'entity_F7': record.get('f7_records'),
          'decision_log': record.get('log'),
          'findings': record.get('findings'),
          'hold_reasons': record.get('hold_reasons'),
      })}</pre>
      <details><summary>Complete raw durable record</summary><pre>{pretty(record)}</pre></details>
    </article>""")

report_html = f"""<!doctype html><html><head><meta charset="utf-8">
<title>CRE machine findings — {esc(RUN_NAME)}</title>
<style>
body{{font-family:Arial,sans-serif;max-width:1200px;margin:30px auto;line-height:1.45}}
.finding{{border:2px solid #333;border-radius:10px;padding:18px;margin:24px 0}}
.cols{{display:grid;grid-template-columns:1fr 1fr;gap:12px}}
pre{{white-space:pre-wrap;overflow-wrap:anywhere;background:#f5f5f5;padding:12px}}
blockquote{{border-left:4px solid #555;padding-left:14px;font-size:1.08em}}
@media(max-width:800px){{.cols{{grid-template-columns:1fr}}}}
</style></head><body>
<h1>CRE F1-F8 machine findings</h1>
<p>Run {esc(RUN_NAME)} · commit {esc(HEAD)} · {len(review_records)} finding records.</p>
<p>This is a machine-decision review artifact, not the blind annotation queue.</p>
{''.join(cards) if cards else '<p>No machine-positive records. Consult reachability before interpreting zero.</p>'}
</body></html>"""
REVIEW_HTML = REVIEW_ROOT / "machine_findings_review.html"
REVIEW_HTML.write_text(report_html, encoding="utf-8")

print("FINDING RECORDS:", len(review_records))
print("REVIEW HTML:", REVIEW_HTML)
display(FileLink(str(REVIEW_HTML)))
if review_records:
    display(HTML(report_html))

## 11. Precision-candidate quotas and blinded human annotation

The target is 10 unique machine-positive candidates per taxonomy across all
completed Drive runs. This is a minimum candidate quota, not a precision result.
Precision is calculated only from rows you explicitly mark graded in the blinded
queue. Machine labels and decisions are kept in a separate key file.

Stop rules:

- stop when every taxonomy has at least 10 unique candidates;
- stop if F8 is the only taxonomy short of 10;
- stop if F2 is the only taxonomy short of 10 and has at most 6;
- otherwise, review the measured cost and run the next disjoint batch only if
  you approve it.

In [ ]:
# Estimated runtime: under one minute.
import pandas as pd

def precision_wilson(k, n, z=1.959963984540054):
    if not n:
        return (None, None)
    phat = k / n
    den = 1 + z*z/n
    center = (phat + z*z/(2*n)) / den
    half = z * math.sqrt(phat*(1-phat)/n + z*z/(4*n*n)) / den
    return (max(0, center-half), min(1, center+half))

def measured_event_cost(row):
    return sum(int(row.get(key) or 0) * price
               for key, price in PRICE_USD_PER_MTOK.items()) / 1_000_000

QUOTA_PER_TAXONOMY = 10
PRECISION_ROOT = DRIVE_ROOT / "precision_annotation"
PRECISION_ROOT.mkdir(parents=True, exist_ok=True)

all_candidate_items = []
for path in sorted(RUNS_ROOT.glob(
        "*/review/machine_findings_full_records.jsonl")):
    run_root = path.parents[1]
    manifest_path = (run_root / "production_output" / "judgment" /
                     "judgment_run_manifest.json")
    if not manifest_path.exists():
        continue
    try:
        prior_manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    except Exception:
        continue
    if prior_manifest.get("status") != "complete":
        continue
    for item in read_jsonl(path):
        item["source_run"] = run_root.name
        all_candidate_items.append(item)

# One candidate counts once per (taxonomy, citation_id), even if a paper was
# accidentally processed in two runs. Multi-label records may count once for each
# genuinely emitted machine taxonomy.
by_taxonomy_key = {}
by_citation = {}
for item in all_candidate_items:
    record = item.get("record") or {}
    citation_id = str(record.get("citation_id") or "")
    if not citation_id:
        continue
    labels = [label for label in item.get("labels") or []
              if re.fullmatch(r"F[1-8]", str(label))]
    for label in labels:
        by_taxonomy_key.setdefault((label, citation_id), item)
    merged = by_citation.setdefault(citation_id, {
        "labels": set(), "record": record, "source_runs": set()})
    merged["labels"].update(labels)
    merged["source_runs"].add(item.get("source_run"))

quota_counts = collections.Counter(label for label, _ in by_taxonomy_key)
quota_rows = [{"taxonomy": label, "unique_candidates": quota_counts[label],
               "target": QUOTA_PER_TAXONOMY,
               "remaining": max(0, QUOTA_PER_TAXONOMY-quota_counts[label])}
              for label in [f"F{i}" for i in range(1, 9)]]
quota_df = pd.DataFrame(quota_rows) if "pd" in globals() else None

missing = [row["taxonomy"] for row in quota_rows if row["remaining"] > 0]
if not missing:
    QUOTA_DECISION = "STOP_ALL_QUOTAS_MET"
    QUOTA_REASON = "Every F1-F8 taxonomy has at least 10 unique candidates."
elif missing == ["F8"]:
    QUOTA_DECISION = "STOP_F8_ONLY_EXCEPTION"
    QUOTA_REASON = "F8 alone is short; use the existing F8 evaluation examples."
elif missing == ["F2"] and quota_counts["F2"] <= 6:
    QUOTA_DECISION = "STOP_F2_ONLY_LOW_YIELD_EXCEPTION"
    QUOTA_REASON = (
        f"F2 alone is short and has {quota_counts['F2']} candidates (at most 6); "
        "use the existing F2 evaluation examples.")
else:
    QUOTA_DECISION = "OPTIONAL_NEXT_DISJOINT_BATCH"
    QUOTA_REASON = "One or more non-exempt taxonomy quotas remain below 10."

quota_payload = {"updated_at": utc_now(), "target": QUOTA_PER_TAXONOMY,
                 "counts": dict(quota_counts), "rows": quota_rows,
                 "decision": QUOTA_DECISION, "reason": QUOTA_REASON,
                 "deduplication_key": "taxonomy + citation_id"}
atomic_json(PRECISION_ROOT / "quota_status.json", quota_payload)

# Build one blinded row per unique citation. The machine-label key is a separate
# file. Existing human entries survive regeneration by blind_id.
QUEUE_PATH = PRECISION_ROOT / "precision_blinded_queue.csv"
KEY_PATH = PRECISION_ROOT / "precision_machine_key.jsonl"
existing_annotations = {}
if QUEUE_PATH.exists():
    try:
        old = pd.read_csv(QUEUE_PATH, dtype=str).fillna("")
        existing_annotations = {row["blind_id"]: row for row in old.to_dict("records")}
    except Exception:
        existing_annotations = {}

blind_rows = []
key_rows = []
for citation_id, item in sorted(by_citation.items()):
    record = item["record"]
    blind_id = "B" + sha256_bytes(citation_id.encode())[:12]
    prior = existing_annotations.get(blind_id, {})
    claimed = record.get("cited_claimed") or record.get("claimed") or {}
    retrieved = record.get("retrieved") or {}
    links = dict(paper_links(record) + related_paper_links(record))
    blind_rows.append({
        "blind_id": blind_id,
        "citation_id": citation_id,
        "paper_links_json": json.dumps(links, ensure_ascii=False, sort_keys=True),
        "citing_sentence": record.get("citing_sentence") or record.get("citance") or "",
        "claimed_reference_json": json.dumps(claimed, ensure_ascii=False, sort_keys=True),
        "retrieved_reference_json": json.dumps(retrieved, ensure_ascii=False, sort_keys=True),
        "atomic_claims_json": json.dumps(record.get("atomic_claims") or [],
                                          ensure_ascii=False),
        "exact_source_spans_json": json.dumps(exact_spans(record),
                                                ensure_ascii=False),
        "graded": prior.get("graded", ""),
        "human_taxonomies": prior.get("human_taxonomies", ""),
        "human_notes": prior.get("human_notes", ""),
        "adjudicator": prior.get("adjudicator", ""),
    })
    key_rows.append({
        "blind_id": blind_id, "citation_id": citation_id,
        "machine_taxonomies": sorted(item["labels"]),
        "source_runs": sorted(item["source_runs"]),
        "record_sha256": sha256_bytes(json.dumps(
            record, sort_keys=True, separators=(",", ":"),
            ensure_ascii=False).encode()),
    })

pd.DataFrame(blind_rows).to_csv(QUEUE_PATH, index=False)
with KEY_PATH.open("w", encoding="utf-8") as fh:
    for row in key_rows:
        fh.write(json.dumps(row, sort_keys=True, ensure_ascii=False) + "\n")

print("PRECISION CANDIDATE QUOTAS")
display(pd.DataFrame(quota_rows))
print("DECISION:", QUOTA_DECISION)
print("REASON:", QUOTA_REASON)
print("BLINDED QUEUE:", QUEUE_PATH)
print("MACHINE KEY (keep separate from annotator):", KEY_PATH)

current_success = [row for row in read_jsonl(MODEL_EVENTS)
                   if row.get("result") == "success"]
current_cost = sum(measured_event_cost(row) for row in current_success)
print("CURRENT BATCH ESTIMATED COST USD:", round(current_cost, 4))
if QUOTA_DECISION == "OPTIONAL_NEXT_DISJOINT_BATCH":
    print("NEXT BATCH IS OPTIONAL, NOT AUTOMATIC.")
    print("If this observed cost is acceptable: increase BATCH_INDEX by 1, "
          "restart the runtime, and rerun from Section 1.")

# Precision is emitted only for explicitly graded rows. The annotator enters a
# comma-separated set such as F3,F6, or leaves human_taxonomies blank after
# setting graded=yes when none of the taxonomies apply.
graded_df = pd.read_csv(QUEUE_PATH, dtype=str).fillna("")
graded_df = graded_df[graded_df["graded"].str.strip().str.casefold().isin(
    {"yes", "true", "1", "graded"})]
key_by_blind = {row["blind_id"]: row for row in key_rows}
precision_rows = []
for label in [f"F{i}" for i in range(1, 9)]:
    sampled = 0
    true_positive = 0
    for row in graded_df.to_dict("records"):
        key = key_by_blind.get(row["blind_id"])
        if not key or label not in key["machine_taxonomies"]:
            continue
        sampled += 1
        human = {part.strip().upper() for part in
                 re.split(r"[,;|]", row["human_taxonomies"]) if part.strip()}
        true_positive += label in human
    low, high = precision_wilson(true_positive, sampled)
    precision_rows.append({
        "taxonomy": label, "graded_machine_positives": sampled,
        "true_positives": true_positive,
        "precision": true_positive/sampled if sampled else None,
        "wilson_95_low": low, "wilson_95_high": high,
        "reportable_minimum_10_graded": sampled >= 10,
    })
precision_df = pd.DataFrame(precision_rows)
precision_df.to_csv(PRECISION_ROOT / "precision_after_human_annotation.csv",
                    index=False)
if len(graded_df):
    print("HUMAN-GRADED PRECISION (rows below 10 are explicitly non-reportable)")
    display(precision_df)
else:
    print("No rows are marked graded yet; no precision number was calculated.")

## 12. Publication measurements and comparisons

These are observed run measurements. Frequencies are not precision. A precision
estimate requires independent blind human labels; the notebook never treats its
own predictions as gold.

In [ ]:
# Estimated runtime: under one minute for ordinary corpora.
import pandas as pd

def percentile(values, p):
    values = sorted(float(x) for x in values)
    if not values:
        return None
    if len(values) == 1:
        return values[0]
    index = (len(values) - 1) * p
    lo, hi = math.floor(index), math.ceil(index)
    return values[lo] if lo == hi else values[lo] + (values[hi] - values[lo]) * (index - lo)

def wilson(k, n, z=1.959963984540054):
    if not n:
        return (None, None)
    phat = k / n
    den = 1 + z*z/n
    center = (phat + z*z/(2*n)) / den
    half = z * math.sqrt(phat*(1-phat)/n + z*z/(4*n*n)) / den
    return (max(0, center-half), min(1, center+half))

model_events = read_jsonl(MODEL_EVENTS)
success_events = [row for row in model_events if row.get("result") == "success"]
http_events = read_jsonl(HTTP_EVENTS)
cache_events = read_jsonl(CACHE_EVENTS)

def event_cost(row):
    return sum(int(row.get(key) or 0) * price
               for key, price in PRICE_USD_PER_MTOK.items()) / 1_000_000

for row in success_events:
    row["estimated_cost_usd"] = event_cost(row)

taxonomy_metrics = []
for row in reachability_rows:
    label = row["taxonomy"]
    count = finding_counts[label]
    lo, hi = wilson(count, len(band2_rows))
    taxonomy_metrics.append({
        **row, "references_total": len(band2_rows),
        "observed_per_100_references": 100 * count / len(band2_rows) if band2_rows else None,
        "wilson_95_low_per_100": None if lo is None else 100*lo,
        "wilson_95_high_per_100": None if hi is None else 100*hi,
    })
taxonomy_df = pd.DataFrame(taxonomy_metrics)

dispositions = collections.Counter(row.get("disposition") for row in band2_rows)
holds = collections.Counter(reason for row in band2_rows
                            for reason in row.get("hold_reasons") or [])
routes = collections.Counter(str(row.get("route")) for row in band2_rows)

stage_rows = []
for stage in sorted({row.get("stage") for row in success_events}):
    events = [row for row in success_events if row.get("stage") == stage]
    latencies = [row.get("elapsed_s") or 0 for row in events]
    stage_rows.append({
        "stage": stage, "calls": len(events),
        "elapsed_total_s": sum(latencies),
        "elapsed_mean_s": statistics.mean(latencies) if latencies else None,
        "elapsed_median_s": statistics.median(latencies) if latencies else None,
        "elapsed_p90_s": percentile(latencies, .90),
        "input_tokens": sum(int(row.get("input_tokens") or 0) for row in events),
        "output_tokens": sum(int(row.get("output_tokens") or 0) for row in events),
        "cache_creation_tokens": sum(int(row.get("cache_creation_input_tokens") or 0)
                                     for row in events),
        "cache_read_tokens": sum(int(row.get("cache_read_input_tokens") or 0)
                                 for row in events),
        "estimated_cost_usd": sum(row["estimated_cost_usd"] for row in events),
    })
stage_df = pd.DataFrame(stage_rows)

def reference_identity_group(row):
    claimed = row.get("cited_claimed") or {}
    if claimed.get("claimed_pmid"):
        return "printed PMID"
    if claimed.get("claimed_doi"):
        return "DOI, no PMID"
    return "neither printed PMID nor DOI"

comparison_rows = []
preband_label_by_id = {row.get("citation_id"): row.get("label")
                       for row in band1_rows}

def all_run_labels(row):
    labels = set(labels_of(row))
    preband = preband_label_by_id.get(row.get("citation_id"))
    if preband in {"F1", "F2", "F8"}:
        labels.add(preband)
    return sorted(labels)

for dimension, grouper in {
    "reference_identifier": reference_identity_group,
    "citation_group": lambda row: "co-citation" if len(
        row.get("citance_group_members") or []) > 1 else "single citation",
    "fulltext_status": lambda row: "complete cited full text" if
        ((row.get("evidence") or {}).get("cited_fulltext") or {}).get(
            "retrieval_complete") is True else "missing/incomplete cited full text",
    "judgment_route": lambda row: str(row.get("route")),
}.items():
    groups = collections.defaultdict(list)
    for row in band2_rows:
        groups[grouper(row)].append(row)
    for group, rows_in_group in sorted(groups.items()):
        labels = collections.Counter(label for item in rows_in_group
                                     for label in all_run_labels(item))
        comparison_rows.append({
            "dimension": dimension, "group": group, "references": len(rows_in_group),
            **{f"{label}_count": labels[label] for label in [f"F{i}" for i in range(1, 9)]},
            **{f"{label}_per_100": 100*labels[label]/len(rows_in_group)
               for label in [f"F{i}" for i in range(1, 9)]},
        })
comparison_df = pd.DataFrame(comparison_rows)

paper_ref_counts = [row["references"] for row in profiles]
total_cost = sum(row["estimated_cost_usd"] for row in success_events)
total_findings = sum(finding_counts.values())
cache_outcomes = collections.Counter((row.get("kind"), row.get("result"))
                                     for row in cache_events)
http_latencies = [float(row.get("elapsed_s") or 0) for row in http_events]

summary = {
    "schema": "cre_f1_f8_publication_measurements_v1",
    "run_name": RUN_NAME, "code_commit": HEAD,
    "corpus": {
        "selection": selection, "papers": len(PMCIDS), "references": len(band2_rows),
        "references_per_paper_mean": statistics.mean(paper_ref_counts),
        "references_per_paper_median": statistics.median(paper_ref_counts),
        "references_per_paper_min": min(paper_ref_counts),
        "references_per_paper_max": max(paper_ref_counts),
        "references_per_paper_p25": percentile(paper_ref_counts, .25),
        "references_per_paper_p75": percentile(paper_ref_counts, .75),
    },
    "taxonomy_findings": dict(finding_counts),
    "dispositions": dict(dispositions), "hold_reasons": dict(holds),
    "routes": dict(routes),
    "runtime": {
        "wall_s": RUN_ELAPSED_S,
        "seconds_per_paper": RUN_ELAPSED_S/len(PMCIDS),
        "seconds_per_reference": RUN_ELAPSED_S/len(band2_rows),
        "references_per_hour": len(band2_rows)/(RUN_ELAPSED_S/3600),
        "http_requests_logged": len(http_events),
        "http_latency_median_s": statistics.median(http_latencies) if http_latencies else None,
        "http_latency_p90_s": percentile(http_latencies, .90),
    },
    "model_and_cost": {
        "successful_calls": len(success_events),
        "failed_calls": sum(row.get("result") != "success" for row in model_events),
        "input_tokens": sum(int(row.get("input_tokens") or 0) for row in success_events),
        "output_tokens": sum(int(row.get("output_tokens") or 0) for row in success_events),
        "cache_creation_input_tokens": sum(int(row.get("cache_creation_input_tokens") or 0)
                                           for row in success_events),
        "cache_read_input_tokens": sum(int(row.get("cache_read_input_tokens") or 0)
                                       for row in success_events),
        "estimated_cost_usd": total_cost,
        "estimated_cost_per_paper_usd": total_cost/len(PMCIDS),
        "estimated_cost_per_reference_usd": total_cost/len(band2_rows),
        "estimated_cost_per_finding_usd": total_cost/total_findings if total_findings else None,
        "linear_projection_1000_references_usd":
            total_cost/len(band2_rows)*1000,
        "price_table_usd_per_mtok": PRICE_USD_PER_MTOK,
        "price_source": PRICE_SOURCE, "price_checked_date": PRICE_CHECKED_DATE,
        "warning": "Estimated from returned token usage; reconcile to provider invoice.",
    },
    "cache_outcomes": {"|".join(map(str, key)): value
                       for key, value in cache_outcomes.items()},
    "precision_candidate_quota": quota_payload,
    "interpretation": (
        "Observed frequencies describe this corpus and are not precision. Precision "
        "requires independent blind human labels. Query-sample results also depend "
        "on the saved search pool and deterministic draw."),
}

atomic_json(MEASURE_ROOT / "publication_measurements.json", summary)
taxonomy_df.to_csv(MEASURE_ROOT / "taxonomy_frequency_and_reachability.csv", index=False)
stage_df.to_csv(MEASURE_ROOT / "model_stage_time_tokens_cost.csv", index=False)
comparison_df.to_csv(MEASURE_ROOT / "comparison_groups.csv", index=False)
pd.DataFrame(profiles).to_csv(MEASURE_ROOT / "paper_input_profile.csv", index=False)
pd.DataFrame([{"disposition": k, "count": v} for k, v in dispositions.items()]).to_csv(
    MEASURE_ROOT / "dispositions.csv", index=False)
pd.DataFrame([{"hold_reason": k, "count": v} for k, v in holds.items()]).to_csv(
    MEASURE_ROOT / "hold_reasons.csv", index=False)

print("\nTAXONOMY FREQUENCY + REACHABILITY")
display(taxonomy_df)
print("\nMODEL STAGE TIME, TOKENS, AND ESTIMATED COST")
display(stage_df)
print("\nKEY COMPARISONS")
display(comparison_df)
print("\nSUMMARY")
print(json.dumps(summary, indent=2, ensure_ascii=False))

## 13. Package the review and measurement artifacts

Raw production records and caches already remain in Drive. This creates a small
shareable ZIP containing the reviewer report, measurement tables, manifest,
corpus selection, and full machine-positive records. It does not include secrets.

In [ ]:
# Estimated runtime: under one minute.
EXPORT_DIR = RUN_ROOT / "shareable_export"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
for source in [
    REVIEW_HTML, REVIEW_JSONL, MANIFEST_PATH, CORPUS_ROOT / "selection.json",
    MEASURE_ROOT / "publication_measurements.json",
    MEASURE_ROOT / "taxonomy_frequency_and_reachability.csv",
    MEASURE_ROOT / "model_stage_time_tokens_cost.csv",
    MEASURE_ROOT / "comparison_groups.csv",
    MEASURE_ROOT / "paper_input_profile.csv",
    MEASURE_ROOT / "dispositions.csv", MEASURE_ROOT / "hold_reasons.csv",
    PRECISION_ROOT / "quota_status.json",
    PRECISION_ROOT / "precision_after_human_annotation.csv",
]:
    shutil.copy2(source, EXPORT_DIR / source.name)

zip_path = shutil.make_archive(str(RUN_ROOT / f"{RUN_NAME}_review_bundle"),
                               "zip", root_dir=EXPORT_DIR)
bundle_manifest = {
    path.name: {"sha256": sha256_file(path), "bytes": path.stat().st_size}
    for path in sorted(EXPORT_DIR.iterdir()) if path.is_file()
}
atomic_json(RUN_ROOT / "shareable_export_manifest.json", bundle_manifest)

print("DONE")
print("Full durable run:", RUN_ROOT)
print("Reviewer report:", REVIEW_HTML)
print("Measurements:", MEASURE_ROOT)
print("Shareable bundle:", zip_path)
display(FileLink(zip_path))

## Measurements produced for review

- Per-taxonomy configured/reached/findings funnel, observed frequency per 100
  references, and Wilson 95% intervals.
- Papers, references, mean/median/min/max/interquartile references per paper,
  PMID/DOI availability, co-citation frequency, XML size, and missing citances.
- Disposition, hold-reason, route, coverage/full-text, and quarantine distributions.
- Comparisons by identifier availability, single citation vs co-citation,
  complete vs incomplete cited full text, and judgment route.
- Wall time, seconds per paper/reference, references per hour, HTTP latency,
  model latency by stage, and model-call failures.
- Input/output/cache-write/cache-read tokens by stage; estimated total cost,
  cost per paper/reference/finding, cache outcomes, and an explicitly linear
  1,000-reference projection.

Do not report the observed-frequency table as precision. For precision, grade
the separate blind annotation queue without machine verdicts, then join labels
only after adjudication.